In [1]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES (Run this FIRST, restart runtime if needed)
# ============================================================
import subprocess
import sys

def install_if_missing(package, import_name=None):
    """Install a pip package if it's not already importable."""
    name = import_name or package.replace('-', '_')
    try:
        __import__(name)
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', package],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print(f"   ✅ {package} installed")

install_if_missing('librosa')
install_if_missing('soundfile')
install_if_missing('matplotlib')
install_if_missing('scikit-learn', 'sklearn')
install_if_missing('pandas')
install_if_missing('numba')  # Required by librosa

import os
import gc
import glob
import random
import warnings
import shutil
import numpy as np

# Suppress noisy warnings before imports
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF info/warning logs

import librosa
import soundfile as sf
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from collections import Counter

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

# ── Detect Colab environment ────────────────────────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    matplotlib.use('Agg')

# ── Handle Keras 2 vs Keras 3 (TF 2.15 vs 2.16+) ──────────
# Colab may have TF 2.15 (uses tf.keras) or TF 2.16+ (uses standalone keras 3)
try:
    from tensorflow import keras
    from tensorflow.keras import layers, callbacks, regularizers
except Exception:
    import keras
    from keras import layers, callbacks, regularizers

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ Keras:      {keras.__version__}")
print(f"✅ Librosa:    {librosa.__version__}")
print(f"✅ GPU:        {tf.config.list_physical_devices('GPU') or 'None (CPU mode)'}")
print(f"✅ Colab:      {IN_COLAB}")

✅ TensorFlow: 2.19.0
✅ Keras:      3.13.2
✅ Librosa:    0.11.0
✅ GPU:        [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Colab:      True


DATA LOADING

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vishnu0399/emergency-vehicle-siren-sounds")

print("Path to dataset files:", path)

100%|██████████| 282M/282M [00:07<00:00, 39.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/vishnu0399/emergency-vehicle-siren-sounds/versions/4


In [3]:
class Config:
    # Audio processing
    SAMPLE_RATE = 22050
    DURATION = 3.0
    N_MELS = 128
    N_FFT = 2048
    HOP_LENGTH = 512
    FMIN = 200
    FMAX = 8000

    # Computed
    N_SAMPLES = int(SAMPLE_RATE * DURATION)
    N_TIME_STEPS = int(np.ceil(N_SAMPLES / HOP_LENGTH)) + 1
    INPUT_SHAPE = (N_MELS, N_TIME_STEPS, 1)

    # Training
    BATCH_SIZE = 32
    EPOCHS = 100
    LEARNING_RATE = 1e-3
    PATIENCE = 15
    MIN_LR = 1e-6

    # Augmentation
    AUG_FACTOR = 3
    NOISE_LEVEL = 0.005

    # Paths – all under /content for Colab
    BASE_DIR = "/content"
    SIREN_DIR = "/content/sounds/ambulance/"
    NO_SIREN_DIR = "/content/sounds/traffic/"
    EXPORT_DIR = "/content/siren_detector_v2"

cfg = Config()
print(f"📐 Input shape: {cfg.INPUT_SHAPE}")
print(f"📐 Samples per clip: {cfg.N_SAMPLES}")

📐 Input shape: (128, 131, 1)
📐 Samples per clip: 66150


In [4]:
import glob

print("Ambulance files:", len(glob.glob("/content/sounds/ambulance/*.wav")))
print("Traffic files:", len(glob.glob("/content/sounds/traffic/*.wav")))

Ambulance files: 200
Traffic files: 200


In [6]:
# ============================================================
# CELL 5: DATA AUGMENTATION
# ============================================================
def _safe_time_stretch(y, rate):
    """librosa time_stretch with API compatibility across versions."""
    try:
        # librosa >= 0.10: uses y= keyword
        return librosa.effects.time_stretch(y=y, rate=rate)
    except TypeError:
        # librosa < 0.10: positional arg
        return librosa.effects.time_stretch(y, rate=rate)


def _safe_pitch_shift(y, sr, n_steps):
    """librosa pitch_shift with API compatibility across versions."""
    try:
        # librosa >= 0.10
        return librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)
    except TypeError:
        # librosa < 0.10
        return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)


def augment_audio(y, sr=cfg.SAMPLE_RATE):
    """Apply random audio augmentations to a waveform."""
    augmented = y.copy()
    aug_choices = ['noise', 'time_stretch', 'pitch_shift', 'volume', 'time_shift']
    selected = random.sample(aug_choices, k=random.randint(1, 3))

    for aug in selected:
        try:
            if aug == 'noise':
                noise = np.random.normal(0, cfg.NOISE_LEVEL, len(augmented))
                augmented = augmented + noise

            elif aug == 'time_stretch':
                rate = np.random.uniform(0.85, 1.15)
                augmented = _safe_time_stretch(augmented, rate)

            elif aug == 'pitch_shift':
                n_steps = np.random.uniform(-2, 2)
                augmented = _safe_pitch_shift(augmented, sr, n_steps)

            elif aug == 'volume':
                gain = np.random.uniform(0.5, 1.5)
                augmented = augmented * gain

            elif aug == 'time_shift':
                shift = int(np.random.uniform(-0.5, 0.5) * sr)
                augmented = np.roll(augmented, shift)

        except Exception:
            # If any augmentation fails, skip it quietly
            pass

    # Fix length after time stretching
    target_len = cfg.N_SAMPLES
    if len(augmented) < target_len:
        augmented = np.pad(augmented, (0, target_len - len(augmented)),
                           mode='constant')
    else:
        augmented = augmented[:target_len]

    # Re-normalize
    max_val = np.max(np.abs(augmented))
    if max_val > 0:
        augmented = augmented / max_val

    return augmented


def spec_augment(mel_spec, num_freq_masks=2, num_time_masks=2,
                 freq_mask_param=15, time_mask_param=20):
    """SpecAugment: randomly mask frequency and time bands."""
    spec = mel_spec.copy()
    n_mels, n_time = spec.shape

    for _ in range(num_freq_masks):
        f = np.random.randint(0, freq_mask_param)
        f0 = np.random.randint(0, max(1, n_mels - f))
        spec[f0:f0 + f, :] = 0

    for _ in range(num_time_masks):
        t = np.random.randint(0, time_mask_param)
        t0 = np.random.randint(0, max(1, n_time - t))
        spec[:, t0:t0 + t] = 0

    return spec


print("✅ Augmentation functions ready.")


✅ Augmentation functions ready.


In [5]:
# ============================================================
# CELL 4: AUDIO PROCESSING FUNCTIONS
# ============================================================
def load_audio(file_path, sr=cfg.SAMPLE_RATE, duration=cfg.DURATION):
    """Load audio file with padding/truncation and normalization."""
    try:
        y, _ = librosa.load(file_path, sr=sr, duration=duration, mono=True)
        target_len = int(sr * duration)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)), mode='constant')
        else:
            y = y[:target_len]
        max_val = np.max(np.abs(y))
        if max_val > 0:
            y = y / max_val
        return y
    except Exception as e:
        print(f"  ⚠️ Error loading {os.path.basename(file_path)}: {e}")
        return None


def audio_to_mel_spectrogram(y, sr=cfg.SAMPLE_RATE):
    """Convert waveform → normalized log-mel spectrogram."""
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=cfg.N_MELS, n_fft=cfg.N_FFT,
        hop_length=cfg.HOP_LENGTH, fmin=cfg.FMIN, fmax=cfg.FMAX,
        power=2.0
    )
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    denom = log_mel.max() - log_mel.min() + 1e-8
    log_mel = (log_mel - log_mel.min()) / denom
    return log_mel


def extract_features(file_path):
    """Full pipeline: file → mel spectrogram."""
    y = load_audio(file_path)
    if y is None:
        return None
    return audio_to_mel_spectrogram(y)


print("✅ Audio processing pipeline ready.")
print(f"   Expected spectrogram shape: ({cfg.N_MELS}, ~{cfg.N_TIME_STEPS})")


✅ Audio processing pipeline ready.
   Expected spectrogram shape: (128, ~131)


In [10]:
def build_dataset(siren_dir, no_siren_dir, aug_factor=cfg.AUG_FACTOR):
    """Load all audio → mel spectrograms with augmentation."""
    X, y_labels = [], []
    audio_ext = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac', '.wma'}
    target_time = cfg.N_TIME_STEPS

    siren_files = [
        f for f in sorted(glob.glob(os.path.join(siren_dir, '*.*')))
        if Path(f).suffix.lower() in audio_ext
    ]
    no_siren_files = [
        f for f in sorted(glob.glob(os.path.join(no_siren_dir, '*.*')))
        if Path(f).suffix.lower() in audio_ext
    ]

    print(f"Found {len(siren_files)} siren, {len(no_siren_files)} no-siren files")
    if not siren_files or not no_siren_files:
        raise ValueError("❌ No audio files found! Check your directories.")

    def pad_or_truncate(mel):
        """Ensure consistent time dimension."""
        if mel.shape[1] < target_time:
            mel = np.pad(mel, ((0, 0), (0, target_time - mel.shape[1])),
                         mode='constant')
        elif mel.shape[1] > target_time:
            mel = mel[:, :target_time]
        return mel

    def process_files(file_list, label, label_name):
        count = 0
        print(f"\n{'🚨' if label == 1 else '🔇'} Processing {label_name} clips...")
        for i, fp in enumerate(file_list):
            if (i + 1) % 50 == 0 or (i + 1) == len(file_list):
                print(f"  [{i+1}/{len(file_list)}]")

            audio = load_audio(fp)
            if audio is None:
                continue

            # Original
            mel = pad_or_truncate(audio_to_mel_spectrogram(audio))
            X.append(mel)
            y_labels.append(label)
            count += 1

            # Augmented copies
            for _ in range(aug_factor):
                aug_audio = augment_audio(audio)
                aug_mel = pad_or_truncate(audio_to_mel_spectrogram(aug_audio))
                aug_mel = spec_augment(aug_mel)
                X.append(aug_mel)
                y_labels.append(label)
                count += 1

        return count

    n_siren = process_files(siren_files, 1, "SIREN")
    n_nosiren = process_files(no_siren_files, 0, "NO-SIREN")

    X_arr = np.array(X, dtype=np.float32)
    y_arr = np.array(y_labels, dtype=np.int32)

    # Add channel dimension: (N, mel, time) → (N, mel, time, 1)
    X_arr = X_arr[..., np.newaxis]

    print(f"\n✅ Dataset built!")
    print(f"   Total samples: {len(X_arr)} (siren: {n_siren}, no-siren: {n_nosiren})")
    print(f"   Shape: {X_arr.shape}")
    print(f"   Class distribution: {Counter(y_arr)}")

    return X_arr, y_arr

In [13]:
import glob

print("Ambulance files:", len(glob.glob("/content/sounds/ambulance/*.wav")))
print("Traffic files:", len(glob.glob("/content/sounds/traffic/*.wav")))

Ambulance files: 200
Traffic files: 200


In [14]:
X, y = build_dataset("/content/sounds/ambulance/", "/content/sounds/traffic/")

Found 200 siren, 200 no-siren files

🚨 Processing SIREN clips...
  [50/200]
  [100/200]
  [150/200]
  [200/200]

🔇 Processing NO-SIREN clips...
  [50/200]
  [100/200]
  [150/200]
  [200/200]

✅ Dataset built!
   Total samples: 1600 (siren: 800, no-siren: 800)
   Shape: (1600, 128, 131, 1)
   Class distribution: Counter({np.int32(1): 800, np.int32(0): 800})


In [17]:
# ============================================================
# CELL 8: TRAIN / VAL / TEST SPLIT
# ============================================================
from sklearn.model_selection import train_test_split

# Train (80%) + Temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Split temp → val (10%) + test (10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train)}")
print(f"Val:   {len(X_val)}")
print(f"Test:  {len(X_test)}")

Train: 1280
Val:   160
Test:  160


In [20]:
# ============================================================
# CELL 9: MODEL
# ============================================================
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=cfg.INPUT_SHAPE),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'AUC', 'Precision', 'Recall']
)

In [21]:
# ============================================================
# CELL 10: TRAINING
# ============================================================
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=8,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        factor=0.5,
        patience=4,
        min_lr=cfg.MIN_LR
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=cfg.BATCH_SIZE,
    callbacks=callbacks
)

Epoch 1/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 61ms/step - AUC: 0.7063 - Precision: 0.7674 - Recall: 0.3609 - accuracy: 0.6258 - loss: 0.6636 - val_AUC: 0.8950 - val_Precision: 0.7865 - val_Recall: 0.8750 - val_accuracy: 0.8188 - val_loss: 0.5727 - learning_rate: 0.0010
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - AUC: 0.8437 - Precision: 0.7692 - Recall: 0.7500 - accuracy: 0.7625 - loss: 0.5100 - val_AUC: 0.9567 - val_Precision: 0.9661 - val_Recall: 0.7125 - val_accuracy: 0.8438 - val_loss: 0.3736 - learning_rate: 0.0010
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - AUC: 0.8973 - Precision: 0.8600 - Recall: 0.8062 - accuracy: 0.8375 - loss: 0.4100 - val_AUC: 0.9659 - val_Precision: 0.9844 - val_Recall: 0.7875 - val_accuracy: 0.8875 - val_loss: 0.2952 - learning_rate: 0.0010
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - AUC: 0.9101 - Precision: 0.8750 - Recall: 0.8313 - accuracy: 0.8562 - loss: 0.3750 - val_AUC: 0.9734 - val_Precision: 0.9851 - val_Recall: 0.8250 - val_

In [23]:
# ============================================================
# CELL 11: EVALUATION
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix

y_pred_proba = model.predict(X_test).ravel()
threshold = 0.3
y_pred = (y_pred_proba >= threshold).astype(int)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94        80
           1       0.96      0.93      0.94        80

    accuracy                           0.94       160
   macro avg       0.94      0.94      0.94       160
weighted avg       0.94      0.94      0.94       160


Confusion Matrix:
[[77  3]
 [ 6 74]]


In [24]:
# ============================================================
# CELL 12: EXPORT
# ============================================================
import os

os.makedirs(cfg.EXPORT_DIR, exist_ok=True)

# Save Keras model
model.save(os.path.join(cfg.EXPORT_DIR, "siren_detector_v2.keras"))

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = os.path.join(cfg.EXPORT_DIR, "siren_detector_v2.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("✅ Model exported:", tflite_path)

Saved artifact at '/tmp/tmp9zujkrcg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 131, 1), dtype=tf.float32, name='keras_tensor_51')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132807846836752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132807846826768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132807846831184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132807846833872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132807846833488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132807846834256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132805683483216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132807846833680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132807846828496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132807846832720: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Model expor